# Final round — Global best model

## Objective
Re-train the **Round 1** and **Round 2** winners (with hyperparameters stored in JSON), compare them on the **same** train/test split, and select the **overall best** regressor for `Final_Performance_Score`.

## Deliverables
- Test predictions: `artifacts/predictions_test_modele_global.csv`
- Serialized pipeline: `artifacts/best_model.joblib`
- Metadata: `artifacts/best_model_meta.json`

**Prerequisite:** run Notebooks 1 & 2 so `round1_best.json` and `round2_best.json` exist in `artifacts/`.

Each **finalist** is fitted and evaluated in its own **step**, with metrics and a short prediction preview shown **before** the head-to-head comparison. Then a **Comparison result** table ranks both finalists, a **verdict** summarizes MSE and R² winners, and **two bar charts** compare **MSE** and **R²** side by side.

After the global winner is selected, a dedicated **improvement step** performs a finer hyperparameter search (robust CV) for that winner and keeps the better version on the test set.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RepeatedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
TEST_SIZE = 0.2
ARTIFACTS = Path("artifacts")
TARGET = "Final_Performance_Score"

DATA_CANDIDATES = [
    Path("data/adaptive_english_learning_performance.csv"),
    Path("data/english_learning_performance.csv"),
    Path("data/dataset.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
WIN = "#c44e52"
SCATTER = "#4c72b0"


def build_round1_pipeline(winner_key: str, params: dict) -> Pipeline:
    if winner_key == "linear_regression":
        est = LinearRegression()
    elif winner_key == "ridge":
        est = Ridge(**params)
    elif winner_key == "lasso":
        est = Lasso(random_state=RANDOM_STATE, max_iter=20_000, **params)
    else:
        raise ValueError(f"winner_key inconnu (round 1): {winner_key}")
    return Pipeline([("scaler", StandardScaler()), ("model", est)])


def build_round2_pipeline(winner_key: str, params: dict) -> Pipeline:
    p = dict(params)
    for k in ("n_estimators", "min_samples_leaf", "min_samples_split", "max_depth"):
        if k in p and p[k] is not None and k != "max_depth":
            p[k] = int(round(p[k]))
    if winner_key == "decision_tree":
        if p.get("max_depth") is not None:
            p["max_depth"] = int(p["max_depth"]) if p["max_depth"] is not None else None
        est = DecisionTreeRegressor(random_state=RANDOM_STATE, **p)
    elif winner_key == "random_forest":
        if "max_depth" in p and p["max_depth"] is not None:
            p["max_depth"] = int(p["max_depth"])
        est = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1, **p)
    elif winner_key == "gradient_boosting":
        if "max_depth" in p and p["max_depth"] is not None:
            p["max_depth"] = int(p["max_depth"])
        est = GradientBoostingRegressor(random_state=RANDOM_STATE, **p)
    else:
        raise ValueError(f"winner_key inconnu (round 2): {winner_key}")
    return Pipeline([("scaler", StandardScaler()), ("model", est)])

In [ ]:
p1 = ARTIFACTS / "round1_best.json"
p2 = ARTIFACTS / "round2_best.json"
if not p1.is_file() or not p2.is_file():
    raise FileNotFoundError(
        "Exécutez d'abord notebook 1 et 2 pour créer round1_best.json et round2_best.json dans artifacts/."
    )

with open(p1, encoding="utf-8") as f:
    meta1 = json.load(f)
with open(p2, encoding="utf-8") as f:
    meta2 = json.load(f)

if meta1.get("random_state") != RANDOM_STATE or meta2.get("random_state") != RANDOM_STATE:
    print("Attention : random_state différent des JSON — le découpage peut ne pas correspondre.")
if meta1.get("test_size") != TEST_SIZE or meta2.get("test_size") != TEST_SIZE:
    print("Attention : test_size différent des JSON.")

display(
    Markdown(
        "### Candidates loaded from previous rounds\n\n"
        f"| Round | Winner | Key hyperparameters (CV-selected) |\n|:---|:---|:---|\n"
        f"| 1 — linear family | **{meta1['winner']}** | `{meta1.get('params', {})}` |\n"
        f"| 2 — trees / boosting | **{meta2['winner']}** | `{meta2.get('params', {})}` |\n"
    )
)

In [ ]:
df = pd.read_csv(DATA_PATH)
id_cols = [c for c in df.columns if str(c).strip().lower() in ("id", "student_id", "student id")]
df = df.drop(columns=id_cols, errors="ignore")

if TARGET not in df.columns:
    raise ValueError(f"Colonne cible '{TARGET}' absente. Colonnes : {list(df.columns)}")

X = df.drop(columns=[TARGET]).select_dtypes(include=[np.number])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
display(Markdown(f"**Train / test:** `{len(X_train)}` / `{len(X_test)}` — same split as rounds 1–2 (`random_state={RANDOM_STATE}`)."))
display(Markdown("*Data preview (numeric features + target):*"))
display(pd.concat([X, y.rename(TARGET)], axis=1).head(5).round(4))

In [ ]:
def _display_finalist_step(title, y_te, pred, params_note=None, n_preview=8):
    mse = mean_squared_error(y_te, pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_te, pred)
    display(Markdown(f"---\n#### {title}"))
    display(pd.DataFrame([{"MSE": mse, "RMSE": rmse, "R²": r2}]).round(4))
    if params_note:
        display(Markdown(f"*Config (from JSON):* `{params_note}`"))
    preview = pd.DataFrame(
        {
            "actual": y_te.iloc[:n_preview].values,
            "predicted": pred[:n_preview],
            "error": pred[:n_preview] - y_te.iloc[:n_preview].values,
        }
    ).round(4)
    display(Markdown(f"*First {n_preview} test rows:*"))
    display(preview)


pipe_a = build_round1_pipeline(meta1["winner_key"], meta1.get("params") or {})
pipe_b = build_round2_pipeline(meta2["winner_key"], meta2.get("params") or {})

label_a = f"Round 1: {meta1['winner']}"
label_b = f"Round 2: {meta2['winner']}"

display(Markdown(f"### Step A — fit & evaluate finalist: `{label_a}`"))
pipe_a.fit(X_train, y_train)
pred_a = pipe_a.predict(X_test)
_display_finalist_step(
    f"Execution result: **{label_a}**",
    y_test,
    pred_a,
    params_note=meta1.get("params") or {},
)

display(Markdown(f"### Step B — fit & evaluate finalist: `{label_b}`"))
pipe_b.fit(X_train, y_train)
pred_b = pipe_b.predict(X_test)
_display_finalist_step(
    f"Execution result: **{label_b}**",
    y_test,
    pred_b,
    params_note=meta2.get("params") or {},
)

mse_a = mean_squared_error(y_test, pred_a)
mse_b = mean_squared_error(y_test, pred_b)
r2_a = r2_score(y_test, pred_a)
r2_b = r2_score(y_test, pred_b)

finalists = pd.DataFrame(
    [
        {"Finalist": label_a, "MSE": mse_a, "RMSE": np.sqrt(mse_a), "R2": r2_a},
        {"Finalist": label_b, "MSE": mse_b, "RMSE": np.sqrt(mse_b), "R2": r2_b},
    ]
).sort_values("MSE")

display(Markdown("## Comparison result — final round (two finalists, **same test set**)"))
final_disp = finalists.copy().reset_index(drop=True)
final_disp.insert(0, "rank", range(1, len(final_disp) + 1))
final_disp = final_disp.rename(columns={"R2": "R²"})
display(final_disp.round({"MSE": 4, "RMSE": 4, "R²": 4}))

_mse_winner = finalists.iloc[0]["Finalist"]
_r2_order = finalists.sort_values("R2", ascending=False)
_r2_winner = _r2_order.iloc[0]["Finalist"]
_dmse = float(abs(mse_a - mse_b))
_dr2 = float(abs(r2_a - r2_b))
display(
    Markdown(
        f"**Verdict (primary — MSE):** **`{_mse_winner}`** wins (lowest test MSE). "
        f"**|ΔMSE|** between finalists = **`{_dmse:.4f}`**. \n"
        f"**Best test R²:** **`{_r2_winner}`** · **|ΔR²|** = **`{_dr2:.4f}`** "
        f"*(if this name differs from the MSE winner, the two metrics disagree on ranking).*"
    )
)

if mse_a <= mse_b:
    winner_pipe = pipe_a
    best_label = label_a
else:
    winner_pipe = pipe_b
    best_label = label_b

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors_mse = [WIN, "#b0b0b0"] if mse_a <= mse_b else ["#b0b0b0", WIN]
colors_r2 = [WIN, "#b0b0b0"] if r2_a >= r2_b else ["#b0b0b0", WIN]
axes[0].bar([label_a, label_b], [mse_a, mse_b], color=colors_mse, edgecolor="white", width=0.55)
axes[0].set_ylabel("Test MSE")
axes[0].set_title("MSE — lower is better (highlight = better)")
axes[1].bar([label_a, label_b], [r2_a, r2_b], color=colors_r2, edgecolor="white", width=0.55)
axes[1].set_ylabel("Test R²")
axes[1].set_title("R² — higher is better (highlight = better)")
for ax in axes:
    ax.tick_params(axis="x", rotation=18)
plt.suptitle("Final round — side-by-side comparison", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

best_mse = float(min(mse_a, mse_b))
best_r2 = float(r2_a if mse_a <= mse_b else r2_b)

display(
    Markdown(
        f"## Overall best model\n\n"
        f"| | |\n|:---|:---|\n"
        f"| **Selected** | **{best_label}** |\n"
        f"| Test MSE | {best_mse:.4f} |\n"
        f"| Test RMSE | {np.sqrt(best_mse):.4f} |\n"
        f"| Test R² | {best_r2:.4f} |\n"
    )
)

In [ ]:
# --- Improvement step: refine the globally selected winner ---
base_pipe = winner_pipe
base_label = best_label
base_pred = base_pipe.predict(X_test)
base_mse = mean_squared_error(y_test, base_pred)
base_r2 = r2_score(y_test, base_pred)

cv_refine = RepeatedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)

if "Ridge" in base_label:
    refine_name = "Ridge (refined with polynomial option)"
    refine_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("poly", PolynomialFeatures(include_bias=False)),
            ("model", Ridge()),
        ]
    )
    refine_grid = {
        "poly__degree": [1, 2],
        "model__alpha": np.logspace(-4, 4, 25),
    }
elif "Lasso" in base_label:
    refine_name = "Lasso (refined with polynomial option)"
    refine_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("poly", PolynomialFeatures(include_bias=False)),
            ("model", Lasso(random_state=RANDOM_STATE, max_iter=50_000)),
        ]
    )
    refine_grid = {
        "poly__degree": [1, 2],
        "model__alpha": np.logspace(-5, 1, 30),
    }
elif "Gradient Boosting" in base_label:
    refine_name = "Gradient Boosting (refined)"
    refine_pipe = GradientBoostingRegressor(random_state=RANDOM_STATE)
    refine_grid = {
        "n_estimators": [150, 200, 300, 400],
        "learning_rate": [0.03, 0.05, 0.08, 0.1],
        "max_depth": [2, 3, 4],
        "min_samples_leaf": [1, 2, 4],
    }
elif "Random Forest" in base_label:
    refine_name = "Random Forest (refined)"
    refine_pipe = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)
    refine_grid = {
        "n_estimators": [200, 300, 500],
        "max_depth": [None, 8, 12, 16],
        "min_samples_leaf": [1, 2, 4],
    }
else:
    refine_name = "Decision Tree (refined)"
    refine_pipe = DecisionTreeRegressor(random_state=RANDOM_STATE)
    refine_grid = {
        "max_depth": [3, 5, 8, 12, None],
        "min_samples_leaf": [1, 2, 4, 6],
        "min_samples_split": [2, 5, 10],
    }

refiner = GridSearchCV(
    estimator=refine_pipe,
    param_grid=refine_grid,
    cv=cv_refine,
    scoring="neg_mean_squared_error",
    n_jobs=1,
)
refiner.fit(X_train, y_train)
refined_pipe = refiner.best_estimator_
refined_pred = refined_pipe.predict(X_test)
refined_mse = mean_squared_error(y_test, refined_pred)
refined_r2 = r2_score(y_test, refined_pred)

improv_df = pd.DataFrame(
    [
        {"Version": "Baseline winner", "MSE": base_mse, "RMSE": np.sqrt(base_mse), "R2": base_r2},
        {"Version": refine_name, "MSE": refined_mse, "RMSE": np.sqrt(refined_mse), "R2": refined_r2},
    ]
).sort_values("MSE")

display(Markdown("## Improvement result — baseline vs refined winner"))
display(improv_df.round(4).reset_index(drop=True))
display(Markdown(f"**Best CV params (refined):** `{refiner.best_params_}`"))

if refined_mse < base_mse:
    winner_pipe = refined_pipe
    best_label = f"{base_label} -> refined"
    y_hat = refined_pred
else:
    y_hat = base_pred

display(Markdown(f"**Selected final version:** `{best_label}`"))

pred_finale = pd.DataFrame(
    {
        "Final_Performance_Score_reel": y_test.values,
        "Final_Performance_Score_pred": y_hat,
    },
    index=y_test.index,
)
pred_finale["erreur"] = pred_finale["Final_Performance_Score_pred"] - pred_finale["Final_Performance_Score_reel"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(y_test, y_hat, alpha=0.55, edgecolors="none", s=28, c=SCATTER)
lo = float(min(y_test.min(), y_hat.min()))
hi = float(max(y_test.max(), y_hat.max()))
axes[0].plot([lo, hi], [lo, hi], "k--", lw=1.2)
axes[0].set_xlabel("Actual score")
axes[0].set_ylabel("Predicted score")
axes[0].set_title(f"Best model: actual vs predicted\n{best_label}")

residuals = y_hat - y_test.values
axes[1].hist(residuals, bins=25, color="#8172b2", edgecolor="white", alpha=0.9)
axes[1].axvline(0, color="black", linestyle="--", lw=1)
axes[1].set_xlabel("Residual (predicted − actual)")
axes[1].set_title("Residuals on test set")

plt.suptitle("Global winner — diagnostic plots", fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

out_csv = ARTIFACTS / "predictions_test_modele_global.csv"
pred_finale.to_csv(out_csv, index=True)
display(Markdown(f"**Test predictions:** `{out_csv}` ({len(pred_finale)} rows) — sample:"))
display(pred_finale.head(12).round(4))

model_path = ARTIFACTS / "best_model.joblib"
joblib.dump(winner_pipe, model_path)
meta_best = {
    "modele": best_label,
    "target": TARGET,
    "colonnes_features": list(X.columns),
    "fichier_modele": model_path.name,
    "random_state": RANDOM_STATE,
    "improvement_step": {
        "baseline_mse": float(base_mse),
        "refined_mse": float(refined_mse),
        "refined_best_params": refiner.best_params_,
    },
}
with open(ARTIFACTS / "best_model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta_best, f, indent=2, ensure_ascii=False)

display(
    Markdown(
        f"**Artifacts written**\n\n"
        f"- Pipeline: `{model_path.resolve()}`\n"
        f"- Metadata: `{(ARTIFACTS / 'best_model_meta.json').resolve()}`\n"
    )
)